In [1]:
import tensorflow as tf
import os
##Select GPU 0 or 1
GPU_INDEX = 2
GPU_MEM = 0.1
##GPU Configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
gpu = '/gpu:'+ str(GPU_INDEX)
#Set Memory limit (DO NOT OVER 5G!!!)
tf.config.experimental.set_virtual_device_configuration(gpus[GPU_INDEX], 
                [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=GPU_MEM*40*1024)])
#WARNING!!!
'''You have to put some critical codes (e.g. Create ANN and fit)  
    inside the with statement(e.g. with tf.device(gpu):)!!!''';
#tf.config.experimental.set_visible_devices([], 'GPU')
#mirrored_strategy = tf.distribute.MirroredStrategy()

In [2]:
import numpy as np
from tensorflow import keras
import copy
import numpy as np
import pickle

path = '/LOCAL/dvs_dataset/cifar10_dvs_shuffle_1300ms_S1_N1.npz'

with np.load(path) as data:
    x_train = data['x_train']
    y_train = data['y_train']
    x_test = data['x_test']
    y_test = data['y_test']
    xmax = np.max(x_train)
    x_train = x_train/xmax
    x_test = x_test/xmax
    y_train = keras.utils.to_categorical(y_train,10)
    y_test = keras.utils.to_categorical(y_test,10)
    
with tf.device(gpu):
    tf.random.set_seed(1000)
    train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
    #AUGMENTATION

    BATCH_SIZE = 128
    SHUFFLE_BUFFER_SIZE = 1000
    train_ds = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE)
    val_ds = test_dataset.batch(BATCH_SIZE)

    #Data Augmentation
    data_augmentation = tf.keras.Sequential([
      tf.keras.layers.experimental.preprocessing.RandomTranslation(0.2,0.2,fill_mode='nearest'),
    ])

    train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
from tensorflow import keras
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential,save_model
from tensorflow.keras.layers import Dense, Activation, Flatten, InputLayer
from tensorflow.keras.layers import Conv2D, AveragePooling2D, BatchNormalization
from tensorflow.keras import optimizers
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers
from tensorflow.keras import initializers
from spkeras.layers import  Regularizer,CFS,Clip
from spkeras.models import cnn_to_snn
    
class train_model:
    def __init__(self,train_ds,val_ds,train=True):
        self.num_classes = 10
        self.weight_decay = 0.0005
        self.x_shape = [128,128,2]
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.model = self.build_model()
        
        if train:
            self.model = self.train(self.model)

    def extract_model(self):
        return self.model

    def build_model(self):
        # Build the network of vgg for 10 classes with massive dropout and weight decay as described in the paper.
        use_bias = True
        model = Sequential()
        weight_decay = self.weight_decay
        kernel_regularizer=regularizers.l2(weight_decay)
        model.add(InputLayer(input_shape=self.x_shape))                    
        model.add(Conv2D(64, (8, 8), strides=(4,4), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer))        
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(64, (3, 3),  padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer))        
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))        
        model.add(Conv2D(128, (3, 3), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))        
        model.add(Conv2D(256, (3, 3), strides=(2,2), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         ))
        model.add(BatchNormalization())        
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(256, (3, 3), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0295, seed=None),
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(512, (3, 3),strides=(2,2), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0208, seed=None),
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(512, (3, 3), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0208, seed=None),
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(512, (3, 3), strides=(2,2), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0208, seed=None),
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Conv2D(512, (3, 3), padding='same',use_bias=use_bias,
                         kernel_regularizer=kernel_regularizer
                         #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0208, seed=None),
                         ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(AveragePooling2D(pool_size=(2, 2),strides=(2,2)))
        model.add(Flatten())
        model.add(Dense(512,use_bias=use_bias,
                        kernel_regularizer=kernel_regularizer
                        #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.0625, seed=None),
                        ))
        model.add(BatchNormalization())
        model.add(Clip())    
        model.add(Activation('relu'))
        model.add(Dense(self.num_classes,use_bias=use_bias,
                        kernel_regularizer=kernel_regularizer
                        #kernel_initializer=initializers.RandomNormal(mean=0.0, stddev=0.1414, seed=None),
                        ))
        model.add(Activation('softmax'))
        #model = CustomFit(model)
        return model

    def train(self,model):
        #training parameters
        batch_size = 128
        maxepoches = 300
        learning_rate = 0.1
        
        # The data, shuffled and split between train and test sets:
        lr_decayed_fn = tf.keras.optimizers.schedules.CosineDecay(learning_rate,maxepoches) 
        reduce_lr = keras.callbacks.LearningRateScheduler(lr_decayed_fn)
        
        train = self.train_ds
        val = self.val_ds       


        sgd = optimizers.SGD(learning_rate=learning_rate, momentum=0.9)#, nesterov=True,decay=lr_decay)
        model.compile(loss='categorical_crossentropy', optimizer=sgd,metrics=['accuracy'])
            
            
        #training process in a for loop with learning rate drop every 25 epoches.
        historytemp = model.fit(train,
                                epochs=maxepoches,
                                batch_size=batch_size,
                                validation_data=val,verbose=2,callbacks=[reduce_lr])                
        
        self.model = model
        return model

if __name__ == '__main__':

    with tf.device(gpu):
        model = train_model(train_ds,val_ds)
        mdl = model.extract_model()
        save_model(mdl,'./cifar10_vgg_clip_1.h5')


Epoch 1/300
